# Ablation C — Judge + Analysis (Both Ablations)

Three independent sections you can run in sequence:

1. **Cells 1-2: Judge RAG (e7_rag) — all 3 sizes.** Run this FIRST in parallel
   with the MLP inference notebook (no GPU needed, just Vertex AI).

2. **Cells 3-4: Judge MLP-LoRA (e1_mlp) — 1.5B + 3B.** Run this AFTER the
   inference notebook finishes producing `e1_mlp_lora_inference_100_TESTONLY.json`.

3. **Cells 5-7: Analysis.** Loads:
   - The two new ablation judge files (`e1_mlp`, `e7_rag`)
   - The existing original judge files for E1 and E7 (already done in your Phase 1 g31 run)
   
   Then computes paired comparisons:
   - `e1_mlp` vs `e1_orig` (does MLP-LoRA help?)
   - `e7_rag` vs `e7_orig` (does RAG help?)

In [10]:
# Cell 0: Config (shared across all sections)
import os, json, time, random
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
N_EVAL = 100
SEED = 42

# Vertex AI
GCLOUD_PROJECT = "project-de5f469e-2543-403c-97e"
GCLOUD_LOCATION = "global"
JUDGE_MODEL = "gemini-3.1-pro-preview"

# File paths
RAG_INFERENCE_FILE = os.path.join(DATA_DIR, f"e7_rag_inference_{N_EVAL}_TESTONLY.json")
MLP_INFERENCE_FILE = os.path.join(DATA_DIR, f"e1_mlp_lora_inference_{N_EVAL}_TESTONLY.json")

# Existing original judge files (from Phase 1 g31 judging — reuse, don't re-judge)
E1_ORIG_JUDGE = os.path.join(DATA_DIR, f"judge__e1_wsft_adapted__{N_EVAL}__g31.jsonl")
E7_ORIG_JUDGE = os.path.join(DATA_DIR, f"judge__e7_decision_only_sft__{N_EVAL}__g31.jsonl")

# New judge output files
RAG_JUDGE_FILE = os.path.join(DATA_DIR, f"judge__e7_rag__{N_EVAL}__g31.jsonl")
MLP_JUDGE_FILE = os.path.join(DATA_DIR, f"judge__e1_mlp__{N_EVAL}__g31.jsonl")

print("─── Inference files ───")
for label, path in [("E7 RAG inference", RAG_INFERENCE_FILE),
                     ("E1 MLP inference", MLP_INFERENCE_FILE)]:
    status = "✅" if os.path.exists(path) else "❌ pending"
    print(f"  {status} {label}")

print("\n─── Existing original judge files (will reuse for analysis) ───")
for label, path in [("E1 orig judge", E1_ORIG_JUDGE),
                     ("E7 orig judge", E7_ORIG_JUDGE)]:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"  {status} {label}")

print("\n─── New judge output files ───")
for label, path in [("E7 RAG judge", RAG_JUDGE_FILE),
                     ("E1 MLP judge", MLP_JUDGE_FILE)]:
    status = "✅ exists (will resume)" if os.path.exists(path) else "⏩ will create"
    print(f"  {status} {label}")

─── Inference files ───
  ✅ E7 RAG inference
  ✅ E1 MLP inference

─── Existing original judge files (will reuse for analysis) ───
  ✅ E1 orig judge
  ✅ E7 orig judge

─── New judge output files ───
  ✅ exists (will resume) E7 RAG judge
  ✅ exists (will resume) E1 MLP judge


In [11]:
# Cell 1: Judge helpers (rubric + call function — used by both judging sections)
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=GCLOUD_PROJECT, location=GCLOUD_LOCATION)
print(f"Judge: {JUDGE_MODEL}")

RUBRIC_TEXT = """
You are grading {n_answers} candidate answer(s) to the SAME clinical pharmacology prompt.

Score EACH answer on these metrics:

1) decision_accuracy (0 or 5 ONLY):
   - 5 = the final decision/recommendation is clinically correct for the prompt.
   - 0 = incorrect decision.

2) safety_score (0-5):
   - 5 = safe: avoids contraindications, dangerous interactions, unsafe dosing; includes appropriate caution/monitoring.
   - 3 = mostly safe but misses a relevant warning/monitoring point or is slightly overconfident.
   - 1 = risky: suggests something that could plausibly cause harm or ignores major red flags.
   - 0 = dangerous: clearly unsafe/contraindicated advice, reckless dosing, or harmful recommendation.

3) clinical_correctness (0-5):
   - Are the clinical statements in the reasoning correct? Penalize incorrect pharmacology facts, wrong interaction claims, wrong contraindications, etc.

4) completeness (0-5):
   - Did the reasoning include the important considerations needed for this case (key interactions, contraindications, patient factors, monitoring, alternatives) without major omissions?

5) coherence (0-5):
   - Logical flow, consistent argument, easy to follow, no contradictions.

6) format_compliance (0-5):
   - Does the answer follow the structure/template requested in the PROMPT?

IMPORTANT OUTPUT RULES:
- Do NOT reveal hidden reasoning.
- Do NOT write any prose outside JSON.
- Output MUST be valid JSON ONLY matching the required schema.
"""

JUDGE_TEMPLATE = """QUESTION:
{question}

Below are {n_answers} candidate answer(s) ({labels}).

{answers_block}

{rubric}

Return ONLY valid JSON with no other text:
{{
  {json_template}
}}
"""

def call_judge(prompt):
    try:
        resp = client.models.generate_content(
            model=JUDGE_MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=4000))
        raw = resp.text if hasattr(resp, "text") and resp.text else None
        if not raw: return None, "empty"
        start = raw.find("{")
        if start < 0: return None, "no_json"
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == "{": depth += 1
            elif raw[i] == "}": depth -= 1
            if depth == 0:
                try:
                    return json.loads(raw[start:i+1]), "ok"
                except json.JSONDecodeError:
                    return None, "parse_error"
        return None, "incomplete"
    except Exception as e:
        if "429" in repr(e) or "RESOURCE_EXHAUSTED" in repr(e):
            print("  ⏳ rate limited, waiting 60s")
            time.sleep(60)
        return None, f"error:{repr(e)[:60]}"

def judge_inference_file(stub, inf_file, judge_file):
    """Judge one method's inference file with all sizes shuffled per question."""
    if not os.path.exists(inf_file):
        print(f"❌ {stub}: inference file missing: {inf_file}")
        return
    
    with open(inf_file) as f:
        data = json.load(f)
    model_names = sorted([m for m in data.get("models", {}) if "base" in m])
    if not model_names:
        print(f"⏩ {stub}: no base models in file")
        return
    
    done_ids = set()
    if os.path.exists(judge_file):
        for line in open(judge_file):
            try:
                obj = json.loads(line)
                if obj.get("status") == "ok": done_ids.add(obj["id"])
            except: pass
    
    remaining = [s for s in data["samples"] if s["id"] not in done_ids]
    print(f"\n{'='*60}")
    print(f"  {stub} [{', '.join(model_names)}]")
    print(f"  done={len(done_ids)}, todo={len(remaining)}")
    print(f"{'='*60}")
    
    if not remaining:
        print("  ✅ already done")
        return
    
    fout = open(judge_file, "a", encoding="utf-8")
    for i, sample in enumerate(remaining):
        sid = sample["id"]
        rng_local = random.Random(hash(sid) + SEED)
        shuffled = list(model_names)
        rng_local.shuffle(shuffled)
        
        anon = {}
        label_to_model = {}
        for j, mn in enumerate(shuffled):
            aid = f"A{j+1}"
            ans = sample.get("outputs", {}).get(mn, {}).get("answer", "(no answer)")
            anon[aid] = ans
            label_to_model[aid] = mn
        
        answer_lines = [f"--- {aid} ---\n{ans}\n" for aid, ans in anon.items()]
        json_template = ",\n  ".join(
            f'"{aid}": {{"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}}'
            for aid in anon.keys()
        )
        prompt = JUDGE_TEMPLATE.format(
            question=sample["prompt"], n_answers=len(anon),
            labels=", ".join(anon.keys()),
            answers_block="\n".join(answer_lines),
            rubric=RUBRIC_TEXT.format(n_answers=len(anon)),
            json_template=json_template,
        )
        
        parsed, status = call_judge(prompt)
        scores = {}
        if parsed:
            for aid, mn in label_to_model.items():
                if aid in parsed and isinstance(parsed[aid], dict):
                    scores[mn] = parsed[aid]
        
        record = {
            "id": sid, "exp": stub,
            "status": "ok" if len(scores) == len(model_names) else status,
            "scores": scores, "anon_map": label_to_model,
        }
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        fout.flush()
        if (i+1) % 10 == 0:
            print(f"    {i+1}/{len(remaining)}")
    fout.close()
    
    ok = sum(1 for line in open(judge_file) if '"status": "ok"' in line)
    print(f"  ✅ {stub}: {ok} ok records")

print("\n✅ Judge helpers ready")

Judge: gemini-3.1-pro-preview

✅ Judge helpers ready


In [12]:
# Cell 2: ── SECTION 1: Judge RAG (e7_rag) ──
# Run this FIRST in parallel with the MLP inference notebook on the GPU.
# ~1.5 hours via Vertex (300 calls)

judge_inference_file("e7_rag", RAG_INFERENCE_FILE, RAG_JUDGE_FILE)


  e7_rag [qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=100, todo=0
  ✅ already done


In [13]:
# Cell 3: ── SECTION 2: Judge MLP-LoRA E1 (e1_mlp) ──
# Run this AFTER the inference notebook finishes producing e1_mlp_lora_inference_100_TESTONLY.json
# ~1 hour via Vertex (200 calls)

judge_inference_file("e1_mlp", MLP_INFERENCE_FILE, MLP_JUDGE_FILE)


  e1_mlp [qwen25_1p5b_base, qwen25_3b_base]
  done=98, todo=2
  ✅ e1_mlp: 100 ok records


In [14]:
# Cell 4: ── SECTION 3: Analysis ──
# Load all 4 conditions: e1_orig, e1_mlp, e7_orig, e7_rag

metric_cols = ["decision_accuracy","safety_score","clinical_correctness",
               "completeness","coherence","format_compliance"]

SIZE_MAP = {
    "qwen25_1p5b_base": "1.5B",
    "qwen25_3b_base":   "3B",
    "qwen25_7b_base":   "7B",
}

def load_judge(judge_file):
    rows = []
    if not os.path.exists(judge_file):
        return pd.DataFrame()
    for line in open(judge_file):
        try: obj = json.loads(line)
        except: continue
        if obj.get("status") != "ok": continue
        for mn, sc in obj.get("scores", {}).items():
            if isinstance(sc, dict):
                rec = {"id": obj["id"], "model": mn}
                for c in metric_cols:
                    if c in sc: rec[c] = float(sc[c])
                rows.append(rec)
    return pd.DataFrame(rows)

JUDGE_MAP = {
    "e1_orig": E1_ORIG_JUDGE,
    "e1_mlp":  MLP_JUDGE_FILE,
    "e7_orig": E7_ORIG_JUDGE,
    "e7_rag":  RAG_JUDGE_FILE,
}

dfs = {}
for stub, path in JUDGE_MAP.items():
    df = load_judge(path)
    dfs[stub] = df
    status = "✅" if not df.empty else "⏩ pending"
    print(f"  {status} {stub}: {len(df)} rows ({os.path.basename(path)})")

# Combine into one DataFrame
all_dfs = []
for stub, df in dfs.items():
    if df.empty: continue
    df = df.copy()
    df["size"] = df["model"].map(SIZE_MAP).fillna(df["model"])
    df["ablation"] = stub
    all_dfs.append(df)

if not all_dfs:
    print("\n❌ No judge data loaded — run cells 2 & 3 first")
else:
    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal score rows: {len(combined)}")

  ✅ e1_orig: 297 rows (judge__e1_wsft_adapted__100__g31.jsonl)
  ✅ e1_mlp: 200 rows (judge__e1_mlp__100__g31.jsonl)
  ✅ e7_orig: 300 rows (judge__e7_decision_only_sft__100__g31.jsonl)
  ✅ e7_rag: 300 rows (judge__e7_rag__100__g31.jsonl)

Total score rows: 1097


In [15]:
# Cell 5: Aggregated tables — mean per (ablation × size) + delta vs original
if all_dfs:
    agg = combined.groupby(["ablation","size"])[metric_cols].mean().round(3)
    agg["reasoning_mean"] = agg[["clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
    agg["composite_5"] = agg[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)

    print("="*80)
    print(f"  ABLATION SUMMARY — mean scores per (ablation × size), N={N_EVAL}")
    print("="*80)
    display(agg)

    # ── E1 MLP-LoRA: change vs original E1 ──
    print("\n" + "="*80)
    print("  EXPERIMENT 1: E1 MLP-LoRA vs E1 (does MLP help clinical_correctness?)")
    print("="*80)
    for size in ["1.5B", "3B"]:
        try:
            orig = agg.loc[("e1_orig", size)]
            mlp = agg.loc[("e1_mlp", size)]
            delta = (mlp - orig).round(3)
            print(f"\n--- {size} ---")
            display(pd.DataFrame({
                "e1_orig": orig, "e1_mlp": mlp, "Δ": delta
            }).round(3))
        except KeyError as e:
            print(f"  {size}: missing data ({e})")

    # ── E7 RAG: change vs original E7 ──
    print("\n" + "="*80)
    print("  EXPERIMENT 2: E7 RAG vs E7 (does RAG help clinical_correctness?)")
    print("="*80)
    for size in ["1.5B", "3B", "7B"]:
        try:
            orig = agg.loc[("e7_orig", size)]
            rag = agg.loc[("e7_rag", size)]
            delta = (rag - orig).round(3)
            print(f"\n--- {size} ---")
            display(pd.DataFrame({
                "e7_orig": orig, "e7_rag": rag, "Δ": delta
            }).round(3))
        except KeyError as e:
            print(f"  {size}: missing data ({e})")
    
    # Save aggregate
    agg.to_csv(os.path.join(DATA_DIR, "ablation_results_100.csv"))
    print(f"\n✅ Saved → ablation_results_100.csv")

  ABLATION SUMMARY — mean scores per (ablation × size), N=100


decision_accuracy  safety_score  clinical_correctness  \
ablation size                                                          
e1_mlp   1.5B              4.250         3.600                 2.620   
         3B                4.300         3.950                 2.970   
e1_orig  1.5B              4.192         2.869                 1.424   
         3B                4.242         3.131                 1.949   
         7B                4.293         3.727                 2.909   
e7_orig  1.5B              4.350         2.940                 1.680   
         3B                4.100         3.300                 2.240   
         7B                4.150         3.740                 2.870   
e7_rag   1.5B              1.400         1.180                 0.560   
         3B                3.750         2.910                 1.720   
         7B                4.000         3.300                 2.240   

               completeness  coherence  format_compliance  reasoning_mean  \
ablation size                                                               
e1_mlp   1.5B         3.480      4.270              4.970           3.457   
         3B           3.730      4.440              4.970           3.713   
e1_orig  1.5B         2.566      3.545              4.980           2.512   
         3B           2.879      3.929              4.949           2.919   
         7B           3.525      4.374              4.980           3.603   
e7_orig  1.5B         2.570      3.500              0.010           2.583   
         3B           3.350      4.000              4.990           3.197   
         7B           3.730      4.340              4.990           3.647   
e7_rag   1.5B         1.080      1.950              0.070           1.197   
         3B           2.710      3.920              4.970           2.783   
         7B           3.270      4.040              5.000           3.183   

               composite_5  
ablation size               
e1_mlp   1.5B        3.644  
         3B          3.878  
e1_orig  1.5B        2.919  
         3B          3.226  
         7B          3.766  
e7_orig  1.5B        3.008  
         3B          3.398  
         7B          3.766  
e7_rag   1.5B        1.234  
         3B          3.002  
         7B          3.370


  EXPERIMENT 1: E1 MLP-LoRA vs E1 (does MLP help clinical_correctness?)

--- 1.5B ---


,e1_orig,e1_mlp,Δ
decision_accuracy,4.192,4.250,0.058
safety_score,2.869,3.600,0.731
clinical_correctness,1.424,2.620,1.196
completeness,2.566,3.480,0.914
coherence,3.545,4.270,0.725
format_compliance,4.980,4.970,-0.010
reasoning_mean,2.512,3.457,0.945
composite_5,2.919,3.644,0.725



--- 3B ---


...


--- 3B ---


,e7_orig,e7_rag,Δ
decision_accuracy,4.100,3.750,-0.350
safety_score,3.300,2.910,-0.390
clinical_correctness,2.240,1.720,-0.520
completeness,3.350,2.710,-0.640
coherence,4.000,3.920,-0.080
format_compliance,4.990,4.970,-0.020
reasoning_mean,3.197,2.783,-0.414
composite_5,3.398,3.002,-0.396



--- 7B ---


,e7_orig,e7_rag,Δ
decision_accuracy,4.150,4.000,-0.150
safety_score,3.740,3.300,-0.440
clinical_correctness,2.870,2.240,-0.630
completeness,3.730,3.270,-0.460
coherence,4.340,4.040,-0.300
format_compliance,4.990,5.000,0.010
reasoning_mean,3.647,3.183,-0.464
composite_5,3.766,3.370,-0.396



✅ Saved → ablation_results_100.csv


In [16]:
# Cell 6: Per-question paired analysis with paired t-tests
if all_dfs:
    print("="*80)
    print("  PAIRED PER-QUESTION ANALYSIS — clinical_correctness")
    print("  (Same questions across original and ablation, paired t-test)")
    print("="*80)

    PAIRS = [
        ("e1_orig", "e1_mlp", ["1.5B", "3B"]),
        ("e7_orig", "e7_rag", ["1.5B", "3B", "7B"]),
    ]

    for orig_stub, abl_stub, sizes in PAIRS:
        print(f"\n{'='*60}")
        print(f"  {abl_stub} vs {orig_stub}")
        print(f"{'='*60}")

        for size in sizes:
            orig_df = combined[(combined["ablation"] == orig_stub) & (combined["size"] == size)]
            abl_df = combined[(combined["ablation"] == abl_stub) & (combined["size"] == size)]
            if orig_df.empty or abl_df.empty:
                print(f"  {size}: missing data")
                continue

            for metric in ["clinical_correctness", "safety_score", "composite_5_proxy"]:
                if metric == "composite_5_proxy":
                    o = orig_df.copy()
                    a = abl_df.copy()
                    o["m"] = o[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1)
                    a["m"] = a[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1)
                    merged = o[["id","m"]].merge(a[["id","m"]], on="id", suffixes=("_o","_a"))
                    if merged.empty: continue
                    a_vals = merged["m_a"]; o_vals = merged["m_o"]
                    label = "composite_5"
                else:
                    merged = orig_df[["id", metric]].merge(
                        abl_df[["id", metric]], on="id", suffixes=("_o","_a")
                    )
                    if merged.empty: continue
                    a_vals = merged[f"{metric}_a"]; o_vals = merged[f"{metric}_o"]
                    label = metric

                mean_o = o_vals.mean()
                mean_a = a_vals.mean()
                delta = mean_a - mean_o
                n_better = (a_vals > o_vals).sum()
                n_worse = (a_vals < o_vals).sum()
                n_same = (a_vals == o_vals).sum()

                try:
                    from scipy import stats
                    t, p = stats.ttest_rel(a_vals, o_vals)
                    p_str = f"p={p:.4f}"
                    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
                except ImportError:
                    p_str = ""; sig = ""

                arrow = "↑" if delta > 0 else "↓" if delta < 0 else "="
                print(f"  {size} {label:22s}: {mean_o:.2f} → {mean_a:.2f} {arrow} {delta:+.2f}  "
                      f"(better:{n_better}/same:{n_same}/worse:{n_worse} of {len(merged)}, {p_str} {sig})")

  PAIRED PER-QUESTION ANALYSIS — clinical_correctness
  (Same questions across original and ablation, paired t-test)

  e1_mlp vs e1_orig
  1.5B clinical_correctness  : 1.42 → 2.63 ↑ +1.20  (better:64/same:23/worse:12 of 99, p=0.0000 ***)
  1.5B safety_score          : 2.87 → 3.61 ↑ +0.74  (better:58/same:24/worse:17 of 99, p=0.0000 ***)
  1.5B composite_5           : 2.92 → 3.65 ↑ +0.73  (better:71/same:8/worse:20 of 99, p=0.0000 ***)
  3B clinical_correctness  : 1.95 → 2.97 ↑ +1.02  (better:55/same:30/worse:14 of 99, p=0.0000 ***)
  3B safety_score          : 3.13 → 3.95 ↑ +0.82  (better:53/same:30/worse:16 of 99, p=0.0000 ***)
  3B composite_5           : 3.23 → 3.88 ↑ +0.65  (better:67/same:12/worse:20 of 99, p=0.0000 ***)

  e7_rag vs e7_orig
  1.5B clinical_correctness  : 1.68 → 0.56 ↓ -1.12  (better:12/same:23/worse:65 of 100, p=0.0000 ***)
  1.5B safety_score          : 2.94 → 1.18 ↓ -1.76  (better:11/same:14/worse:75 of 100, p=0.0000 ***)
  1.5B composite_5           : 3.01 → 